# Importing Important Libraries For Extraction and Transformation of Data

In [5]:
import pandas as pd
import numpy as np
import fastf1
import os

## Caching Data from Fast F1

In [6]:
fastf1.Cache.enable_cache("f1_cache")

### Loading Data of 2023,2024,2025 Austrain Grand Prix

In [7]:
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

sessions = {}

for year in range(2023, 2026):
    try:
        session = fastf1.get_session(year, 'Austria', 'R')
        session.load()
        sessions[year] = session
        print(f"{year} loaded successfully")
    except Exception as e:
        print(f"Failed for {year}: {e}")

core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '4', '14', '55', '63', '44', '18', '10', '23', '24', '2', '31', '77', '81', '21', '20', '22', '27']


2023 loaded successfully


core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '81', '55', '44', '1', '27', '11', '20', '3', '10', '16', '31', '18', '22', '23', '77', '24', '14', '2', '4']


2024 loaded successfully


core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '16', '44', '63', '30', '14', '5', '27', '31', '87', '6', '10', '18', '43', '22', '23', '1', '12', '55']


2025 loaded successfully


## Fecthing important Columns

In [8]:
laps={}
for year in range(2023, 2026):
    
    laps[year] = sessions[year].laps[["Driver","LapTime","Sector1Time","Sector2Time","Sector3Time"]].copy()
    

In [9]:
laps[2023].head()

,Driver,LapTime,Sector1Time,Sector2Time,Sector3Time
0,VER,0 days 00:01:17.639000,NaT,0 days 00:00:31.613000,0 days 00:00:25.440000
1,VER,0 days 00:01:55.479000,0 days 00:00:31.698000,0 days 00:00:46.293000,0 days 00:00:37.488000
2,VER,0 days 00:02:04.721000,0 days 00:00:37.877000,0 days 00:00:46.298000,0 days 00:00:40.546000
3,VER,0 days 00:01:09.691000,0 days 00:00:17.618000,0 days 00:00:30.970000,0 days 00:00:21.103000
4,VER,0 days 00:01:10.026000,0 days 00:00:17.716000,0 days 00:00:31.158000,0 days 00:00:21.152000


## Dropping all the Null Values


In [10]:
for year in range (2023,2026):
    laps[year].dropna(inplace=True)

## Converting the Times in Seconds 

In [11]:
for year in range (2023,2026):
    for col in["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
        laps[year][f"{col} (s)"] = laps[year][col].dt.total_seconds()

## Taking Average of all the Times

In [12]:
sector_times = {}

for year in range(2023,2026):
    sector_times[year] = laps[year].groupby("Driver").agg({
        "Sector1Time (s)":"mean",
        "Sector2Time (s)":"mean",
        "Sector3Time (s)":"mean"
    }).reset_index()

## Add New Columns Like Total Sector Time and Year

In [13]:
for year in range(2023, 2026):
    sector_times[year]["TotalSectorTime (s)"]=(
        sector_times[year]["Sector1Time (s)"]+
        sector_times[year]["Sector2Time (s)"]+
        sector_times[year]["Sector3Time (s)"]
    )
    sector_times[year]["Year"] = year

In [14]:
sector_times[2024]

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime (s),Year
0,ALB,18.294725,32.249957,21.819913,72.364594,2024
1,ALO,18.713652,32.270536,21.922333,72.906522,2024
2,BOT,18.433464,32.215638,21.832855,72.481957,2024
3,GAS,18.131386,32.102671,21.843871,72.077929,2024
4,HAM,18.281700,31.798114,21.507329,71.587143,2024
5,HUL,18.218443,32.005843,21.753800,71.978086,2024
6,LEC,18.690686,31.859414,21.560886,72.110986,2024
7,MAG,18.123171,32.078443,21.860200,72.061814,2024
8,NOR,18.091159,31.711476,21.906619,71.709254,2024
9,OCO,18.251257,32.230729,21.709729,72.191714,2024


## Concating all the tables

In [15]:
combined_sector_time = pd.concat(sector_times.values(), ignore_index=True)

In [16]:
combined_sector_time

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime (s),Year
0,ALB,18.579229,32.195086,22.230314,73.004629,2023
1,ALO,18.565886,31.831814,22.112986,72.510686,2023
2,BOT,18.798507,32.332826,22.410145,73.541478,2023
3,DEV,18.660580,32.326725,22.340609,73.327913,2023
4,GAS,18.757571,31.935757,22.148171,72.841500,2023
5,HAM,18.657243,31.985800,22.052229,72.695271,2023
6,HUL,21.874818,34.029273,25.448727,81.352818,2023
7,LEC,18.702700,31.715543,21.859271,72.277514,2023
8,MAG,18.811397,32.598985,22.355382,73.765765,2023
9,NOR,18.510629,31.966657,22.003100,72.480386,2023


## Clean Air Race Pace (Best Time From Practice Session Before Qualifying Round.)

In [17]:
clean_air_race_pace = {
    #McLaren
    "NOR":00,               #Lando Norris 🇬🇧
    "PIA":00,               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER":00,               #Max Verstappen 🇳🇱
    "HAD":00,               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC":00,               #Charles Leclerc 🇲🇨
    "HAM":00,               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS":00,               #Geaorge Russell 🇬🇧
    "ANT":00,               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO":00,               #Fernando Alonso 🇪🇸
    "STR":00,               #Lance Stroll 🇨🇦
    #Williams
    "ALB":00,               #Alexander Albon 🇹🇭
    "SAI":00,               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO":00,               #Esteban Ocon 🇫🇷
    "BEA":00,               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS":00,               #Piette Gasly 🇫🇷
    "COL":00,               #Franco Colapinto 🇦🇷
    #Audi
    "HUL":00,               #Nico Hulkenberg 🇩🇪
    "BOR":00,               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER":00,               #Sergio Perez 🇲🇽
    "BOT":00,               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW":00,               #Liam Lawson 🇳🇿
    "LIN":00                #Arvid Lindblad 🇬🇧
}

## List of the Current Drivers

In [18]:
Driver = [
    #McLaren
    "NOR",               #Lando Norris 🇬🇧
    "PIA",               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER",               #Max Verstappen 🇳🇱
    "HAD",               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC",               #Charles Leclerc 🇲🇨
    "HAM",               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS",               #Geaorge Russell 🇬🇧
    "ANT",               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO",               #Fernando Alonso 🇪🇸
    "STR",               #Lance Stroll 🇨🇦
    #Williams
    "ALB",               #Alexander Albon 🇹🇭
    "SAI",               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO",               #Esteban Ocon 🇫🇷
    "BEA",               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS",               #Piette Gasly 🇫🇷
    "COL",               #Franco Colapinto 🇦🇷
    #Audi
    "HUL",               #Nico Hulkenberg 🇩🇪
    "BOR",               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER",               #Sergio Perez 🇲🇽
    "BOT",               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW",               #Liam Lawson 🇳🇿
    "LIN"                #Arvid Lindblad 🇬🇧

]

## Qualifying Time 

In [19]:
qualifying_2026 = pd.DataFrame({
    "Driver":Driver,
    "QualifyingTime (s)":[
        #McLaren
        00,                 #NOR
        00,                 #PIA
        #Red Bull Racing
        00,                 #VER
        00,                 #HAD
        #Ferrari
        00,                 #LEC
        00,                 #HAM
        #Mercedes
        00,                 #RUS
        00,                 #ANT
        #Aston Martin Aramco
        00,                 #ALO
        00,                 #STR
        #Williams
        00,                 #ALB
        00,                 #SAI
        #Haas
        00,                 #OCO
        00,                 #BEA
        #Alpine
        00,                 #GAS
        00,                 #COL
        #Audi
        00,                 #HUL
        00,                 #BOR
        #Cadillac
        00,                 #PER
        00,                 #BOT
        #Racing Bulls
        00,                 #LAW
        00                  #LIN
    ]
})

## Combining Qualifying Time and Clean Air Race Pace 

In [20]:
qualifying_2026["CleanAirRacePace (s)"] = qualifying_2026["Driver"].map(clean_air_race_pace)

In [21]:
qualifying_2026

,Driver,QualifyingTime (s),CleanAirRacePace (s)
0,NOR,0,0
1,PIA,0,0
2,VER,0,0
3,HAD,0,0
4,LEC,0,0
5,HAM,0,0
6,RUS,0,0
7,ANT,0,0
8,ALO,0,0
9,STR,0,0


## Team Points(Currently Zero Due to Start of Season)

In [22]:
team_points = {
    "McLaren":00,
    "Mercedes":00,
    "Red Bull":00,
    "Williams":00,
    "Ferrari":00,
    "Haas":00,
    "Aston Martin":00,
    "Racing Bulls":00,
    "Alpine":00,
    "Audi":00,
    "Cardillac":00
}

## Scaling the Team Points (For Future Use)

In [24]:
from sklearn.preprocessing import MinMaxScaler

teams = list(team_points.keys())
points = np.array(list(team_points.values())).reshape(-1,1)

scaler = MinMaxScaler()
scaled_points = scaler.fit_transform(points)

team_performance_score = dict(zip(teams, scaled_points.flatten()))